# 05. RFM 개념과 손계산
**목표**: RFM이 무엇인지 이해하고, 작은 예제로 계산부터 세그먼트 분류까지 해 봅니다. (예상 4~5시간)

## RFM이란?
고객의 구매 이력으로 **누가 좋은 고객인지** 점수 매기는 기법입니다.

| 지표 | 질문 | 계산 | 좋은 방향 |
|---|---|---|---|
| **R**ecency | 마지막 구매가 언제? | 기준일 − 마지막 구매일(일) | **작을수록** 좋음 |
| **F**requency | 몇 번 샀나? | 주문 횟수 | 클수록 좋음 |
| **M**onetary | 얼마나 썼나? | 구매 금액 합계 | 클수록 좋음 |

> ⚠️ Recency는 **값이 작을수록 좋다**는 점이 가장 흔한 실수 포인트입니다. 점수를 매길 때 방향을 뒤집어야 합니다.

In [ ]:
import numpy as np
import pandas as pd

## 1. 손계산용 미니 데이터 (고객 5명)

In [ ]:
tx = pd.DataFrame({
    "customer_id": ["A", "A", "A", "B", "B", "C", "D", "D", "D", "D", "E"],
    "order_date": pd.to_datetime([
        "2025-12-20", "2025-12-25", "2025-12-30",
        "2025-10-01", "2025-11-15",
        "2025-06-01",
        "2025-12-28", "2025-12-29", "2025-12-30", "2025-12-31",
        "2025-12-30"]),
    "amount": [50000, 70000, 80000, 30000, 20000, 15000, 100000, 120000, 90000, 150000, 25000],
})
tx

### 기준일 정하기
Recency 기준일은 **데이터의 마지막 거래일(또는 분석 기준일)** 입니다. (미션 4.1) 오늘 날짜를 쓰면 오래된 데이터에서 모든 고객의 Recency가 비정상적으로 커집니다.

보통 마지막 거래일 **다음 날**을 기준일로 쓰기도 합니다. (마지막 날 구매한 고객의 Recency가 0이 되지 않도록)

In [ ]:
snapshot = tx["order_date"].max() + pd.Timedelta(days=1)
print("기준일:", snapshot.date())

## 2. 고객별 R, F, M 계산 (groupby + agg)

In [ ]:
rfm = tx.groupby("customer_id").agg(
    recency=("order_date", lambda s: (snapshot - s.max()).days),
    frequency=("order_date", "count"),
    monetary=("amount", "sum"),
)
rfm

**손으로 확인해 보세요.** 고객 A는 마지막 구매가 12/30, 기준일이 2026-01-01(마지막 거래일 다음 날)이므로 recency = 2일, 주문 3번, 합계 200,000원이어야 합니다.

## 3. 점수 매기기: 분위수 기준
각 지표를 5등분하여 1~5점을 줍니다. (고객이 5명뿐이라 여기서는 **3등분(1~3점)** 으로 연습합니다.)

- `pd.qcut(값, q, labels)`: 같은 인원씩 나눔
- 주의: 값이 같은 고객이 많으면 구간 경계가 겹쳐 오류가 납니다 → `rank(method="first")` 로 순위를 먼저 매겨 해결합니다.

In [ ]:
def score(series: pd.Series, q: int = 3, ascending: bool = True) -> pd.Series:
    """값을 q등분해 1~q점 부여. ascending=True면 값이 클수록 높은 점수."""
    ranked = series.rank(method="first", ascending=ascending)
    return pd.qcut(ranked, q, labels=range(1, q + 1)).astype(int)

rfm["R"] = score(rfm["recency"], ascending=False)   # recency는 작을수록 좋음 → 뒤집기
rfm["F"] = score(rfm["frequency"])
rfm["M"] = score(rfm["monetary"])
rfm

Recency가 가장 작은 고객이 R 점수가 **가장 높게** 나왔는지 확인하세요. (`ascending=False` 가 방향을 뒤집어 줍니다.)

## 4. 세그먼트 분류 (규칙 기반)
점수 조합을 사람이 이해할 수 있는 이름으로 바꿉니다. 아래는 하나의 예시이며, 정답은 없고 **데이터와 비즈니스에 맞게 정의**하면 됩니다.

| 세그먼트 | 조건(예) | 의미 |
|---|---|---|
| VIP | R, F, M 모두 높음 | 최근, 자주, 많이 구매 |
| Loyal | F, M 높고 R은 보통 이상 | 꾸준한 단골 |
| New | R 높고 F 낮음 | 최근 첫 구매 |
| Churned | R 낮음 | 오래전에 떠난 고객 |
| Others | 그 외 | 일반 |

(실제 미션에서는 점수를 5단계로 하고 조건을 더 정교하게 만듭니다.)

In [ ]:
def segment(row) -> str:
    if row["R"] >= 3 and row["F"] >= 3 and row["M"] >= 3:
        return "VIP"
    if row["R"] == 1:
        return "Churned"
    if row["R"] >= 3 and row["F"] == 1:
        return "New"
    if row["F"] >= 2 and row["M"] >= 2:
        return "Loyal"
    return "Others"

rfm["segment"] = rfm.apply(segment, axis=1)
rfm

## 5. 세그먼트별 요약
미션 8장의 "RFM 세분화 결과 예시" 표와 같은 형태를 만듭니다.

In [ ]:
summary = rfm.groupby("segment").agg(
    고객수=("monetary", "count"),
    평균구매금액=("monetary", "mean"),
    평균빈도=("frequency", "mean"),
    평균Recency=("recency", "mean"),
).round(1)
summary

## 6. 인사이트 쓰는 형식 (미션 4.8)
각 인사이트는 **근거 / 실행 / 검증** 3요소가 필요합니다.

> **(근거)** Churned 고객 비율이 20%이며 마지막 구매 후 평균 N일이 지났다.
> **(실행)** Churned 대상 → 복귀 쿠폰 발송 → 재구매율 개선 기대.
> **(검증)** 이탈 사유(가격/품질/경쟁사)를 알려면 설문 데이터가 추가로 필요하다.

지금 단계에서는 위 요약 표를 보고 **스스로 인사이트를 한 개** 써 보는 것으로 충분합니다.

## 7. 자주 하는 실수 체크리스트
- [ ] Recency 방향을 뒤집었는가? (작을수록 좋음)
- [ ] 기준일을 오늘이 아니라 **데이터의 마지막 거래일** 로 잡았는가?
- [ ] 고객 단위로 집계했는가? (주문 단위가 아님)
- [ ] 취소/환불 주문, 금액이 0 이하인 행을 걸러냈는가?
- [ ] `customer_id` 결측 행을 처리했는가?

### ✏️ 직접 해보기
고객 E의 R, F, M 값을 직접 읽어 보고, 왜 이런 세그먼트가 되었는지 한 문장으로 설명하세요.

In [ ]:
# 여기에 직접 작성해 보세요

### ✏️ 직접 해보기
`snapshot` 을 `2026-01-31` 로 바꿔서 다시 계산하면 Recency와 세그먼트가 어떻게 달라질지 예측해 보세요. 그 후 실제로 돌려서 확인하세요.

> 힌트: 위 셀들을 복사해 `snapshot` 만 바꾸면 됩니다.

In [ ]:
# 여기에 직접 작성해 보세요

✅ 다음: `06_oop_basics`